In [51]:
sc.stop()
from pyspark import SparkContext
import findspark
findspark.init()

import numpy as np
import warnings
warnings.filterwarnings("ignore")

sc = SparkContext(master="local[*]", appName="TextFileExample")

DATA_FILE = "botnet_sample_1000.csv"
X_SIZE = 11
Y_SIZE = 1
N_ITER = 10
LEARNING_RATE = 1.5

In [52]:
# def readFile (filename):
# Arguments:
# filename – name of the spam dataset file
# 12 columns: 11 features/dimensions (X) + 1 column with labels (Y)
# Y -- Train labels (0 if normal traffic, 1 if botnet)
# m rows: number of examples (m)
# Returns:
# An RDD containing the data of filename. Each example (row) of the file
# corresponds to one RDD record. Each record of the RDD is a tuple (X,y).
# “X” is an array containing the 11 features (float number) of an example
# “y” is the 12th column of an example (integer 0/1)

def readFile(filename):
    rdd = sc.textFile(filename)
    def map_line(line):
        elements = [float(element) for element in line.split(",")]
        return (np.array(elements[:-1]), int(elements[-1]))
    return rdd.map(map_line)



# def normalize (RDD_Xy):
# Arguments:
# RDD_Xy is an RDD containing data examples. Each record of the RDD is a tuple
# (X,y).
# “X” is an array containing the 11 features (float number) of an example
# “y” is the label of the example (integer 0/1)
# Returns:
# An RDD rescaled to N(0,1) in each column (mean=0, standard deviation=1)
def normalize (RDD_Xy):
    
    rdd_col = RDD_Xy.map(lambda xy: (np.array(xy[:-1], dtype=float), int(xy[-1])))

    n = rdd_col.count()

    sum_vec = rdd_col.map(lambda xy: xy[0]).reduce(lambda a, b: a + b)
    media = sum_vec / n
    
    
    varianza = rdd_col.map(lambda v: (v[0]-media)*(v[0]-media)).reduce(lambda a,b:a+b)/n
    
    std=np.sqrt(varianza)
    #Normalize
    
    norm = rdd_col.map(lambda v: (v[0] - media)/std, v[1])
    
    
    return norm


# def train (RDD_Xy, iterations, learning_rate, lambda_reg):
# Arguments:
# RDD_Xy --- RDD containing data examples. Each record of the RDD is a tuple
# (X,y).
# “X” is an array containing the 11 features (float number) of an example
# “y” is the label of the example (integer 0/1)
# iterations -- number of iterations of the optimization loop
# learning_rate -- learning rate of the gradient descent
# lambda_reg – regularization rate
# Returns:
# A list or array containing the weights “w” and bias “b” at the end of the
# training process


def train(RDD_Xy, iterations, learning_rate):
    
    sigma = lambda z : (1 / (1 + (np.e**(-z))))
    
    W = np.zeros(X_SIZE)
    b = 0
    
    def calculate_dw(rdd, W, b):
        rdd = rdd.map(lambda X, y : np.array([(sigma(W * X + b) - y) * x_i for x_i in X]))
        dw = rdd.reduce(lambda a, b : a + b) / rdd.count()
        return dw
    
    def calculate_db(rdd, W, b):
        rdd = rdd.map(lambda X, y : np.array((sigma(W * X + b) - y)))
        db = rdd.reduce(lambda a, b : a + b) / rdd.count()
        return db
    
    for _ in iterations:
        dw = calculate_dw(RDD_Xy, W, b)
        db = calculate_db(RDD_Xy, W, b)
        W = W - learning_rate * dw
        b = b - learning_rate * db
    
    return W, b


# def accuracy (w, b, RDD_Xy):
# Arguments:
# w -- weights
# b -- bias
# RDD_Xy – RDD containing examples to be predicted
# Returns:
# accuracy -- the number of predictions that are correct divided by the number
# of records (examples) in RDD_xy.
# Predict function can be used for predicting a single example



# def predict (w, b, X):
# Arguments:
# w -- weights
# b -- bias
# X – Example to be predicted
# Returns:
# Y_pred – a value (0/1) corresponding to the prediction of X

def predict(w, b, data):
    sigma = lambda z : (1 / (1 + (np.e**(-z))))
    return data.map(lambda X, y : sigma(w*X + b))




In [53]:
# read data
data = readFile(DATA_FILE)

# standarize
rdd=normalize(data)
first_element = rdd.first()

# Print it
print(first_element)

#W, b = train(data, N_ITER, LEARNING_RATE)
#acc = accuracy(W, b, data)
#print("Accuracy: ", acc)

NameError: name 'v' is not defined